[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/mlsysbook-vi-notebooks/blob/main/vol1-ch06-nn-architectures/01-gia-cua-mot-gia-dinh.ipynb)

# Giá của một giả định: đếm tham số, phép tính và byte

Chương 6 lặp đi lặp lại một câu: một kiến trúc **giả định** một cấu trúc nào đó về dữ liệu, giả định đó
quyết định số tham số, số tham số quyết định lượng dữ liệu phải di chuyển, và lượng dữ liệu đó quyết định
phần cứng nghẽn ở tính toán hay nghẽn ở bộ nhớ. Sổ tay này biến câu đó thành phép đếm. Mỗi con số của
chương được tính lại từ hình dạng tensor, rồi đối chiếu với một module PyTorch thật bằng `numel()`.

**Bạn sẽ làm:**
1. đếm tầng dày đặc của chương và mật độ tính toán (arithmetic intensity) 0.5 FLOP/byte của nó;
2. đếm tầng tích chập: 27 trọng số dùng lại ở 50,176 vị trí, so với khoảng 205 triệu của tầng dày đặc
   tương đương, và xem mỗi chi phí lớn lên ra sao khi ảnh lớn lên;
3. đếm tích chập tách theo chiều sâu của bài toán phát hiện từ khoá (KWS): 73,728 xuống 8,768;
4. đếm ResNet-50 và MobileNetV2 thật của torchvision, không tải trọng số nào: tham số, phép nhân cộng
   dồn (MAC) theo từng loại tầng, chữ ký tải công việc và phổ mật độ tính toán;
5. đo hai mô hình đó trên CPU này, để thấy vì sao ít FLOP hơn không có nghĩa là nhanh hơn theo đúng tỉ lệ.

In [ ]:
#@title Chuẩn bị: thư viện, hạt giống và hai hàm đối chiếu với sách { display-mode: "form" }
# Chạy ô này trước. Nó không cài thêm gì: mọi thư viện dùng ở đây đều có sẵn trên Colab.
import math, os, random, re, sys, time

import matplotlib.pyplot as plt
import numpy as np
import torch

# Số luồng CPU. 0 để thư viện tự chọn (trên Colab là 2).
NB_THREADS = int(os.environ.get("NB_THREADS", "0"))
if NB_THREADS:
    torch.set_num_threads(NB_THREADS)
    try:
        torch.set_num_interop_threads(NB_THREADS)
    except RuntimeError:  # chỉ đặt được một lần mỗi phiên
        pass
    from threadpoolctl import threadpool_limits
    threadpool_limits(NB_THREADS)

SEED = 42


def dat_hat_giong(seed=SEED):
    """Cùng hạt giống thì cùng kết quả, mỗi lần chạy lại."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


dat_hat_giong()
DATA = os.environ.get("NB_DATA", "data")  # nơi tải dữ liệu về
NB_STRICT = os.environ.get("NB_STRICT") == "1"
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3})

# Đọc một con số theo cách bảng số liệu của bài học viết nó (dấu chấm hay dấu phẩy).
import math
import re

_SUP = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹⁻⁺", "0123456789-+")
_MULT = {"tỷ": 1e9, "tỉ": 1e9, "triệu": 1e6, "nghìn": 1e3, "ngàn": 1e3,
         "K": 1e3, "M": 1e6, "B": 1e9}
_NUM = re.compile(
    r"(?<![\w.,])(-?)(\d+(?:[.,]\d+)*)"                      # sign, digits with separators
    r"(?:\s*[×x·]\s*10\s*(?:\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))|[eE]([-+]?\d+)"
    r"|\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))?"                  # or a bare power, 10^{-5}
    r"(\s*%|\s*(?:tỷ|tỉ|triệu|nghìn|ngàn)(?!\w)|\s?[KMB](?![A-Za-z]))?")


def norm(text):
    """The outline's markup and typography reduced to plain text, as both sides compare it."""
    text = text.replace("{,}", ",").replace("**", "").replace("`", "")
    text = re.sub(r"[   ]", " ", text)
    text = re.sub(r"−(?=\d)", "-", text)
    text = re.sub(r"[~≈]", "", text)
    return re.sub(r"\s+", " ", text).strip()


def _readings(digits):
    """(value, decimals) for each way the separators in `digits` can be read."""
    out = []
    groups_ok = lambda parts: all(len(p) == 3 for p in parts[1:])
    for thou, dec in ((",", "."), (".", ",")):
        if thou in digits and dec in digits:
            head, _, tail = digits.rpartition(dec)
            if dec in head or not tail.isdigit() or not groups_ok(head.split(thou)):
                continue
            out.append((float(head.replace(thou, "") + "." + tail), len(tail)))
    for sep in (",", "."):
        other = "." if sep == "," else ","
        if sep in digits and other not in digits:
            parts = digits.split(sep)
            if groups_ok(parts):
                out.append((float("".join(parts)), 0))                    # thousands
            if len(parts) == 2:
                out.append((float(parts[0] + "." + parts[1]), len(parts[1])))  # decimal
    if "." not in digits and "," not in digits:
        out.append((float(digits), 0))
    return out


def candidates(text):
    """Every (value, half_unit) the numbers in `text` can mean.

    half_unit is half of the last printed digit, scaled like the value, so `437,5 KB`
    accepts 437.544. Each number is offered as written and, when a multiplier or `%`
    follows it, scaled by that too, so `8,5 tỉ` cites 8.5 or 8.5e9 and `20 %` cites 20 or 0.2.
    """
    out = []
    for m in _NUM.finditer(norm(text)):
        sign, digits, exp_caret, exp_sup, exp_e, pow_caret, pow_sup, suffix = m.groups()
        exp = exp_caret or (exp_sup.translate(_SUP) if exp_sup else None) or exp_e
        power = pow_caret or (pow_sup.translate(_SUP) if pow_sup else None)
        scale10 = 10.0 ** int(exp) if exp else 1.0
        suffix = (suffix or "").strip()
        extra = 0.01 if suffix == "%" else _MULT.get(suffix)
        for value, decimals in _readings(digits):
            if power:
                # 10^{-5}, 2³²: an exact value, so it is matched to 1 part in 10⁹; a looser
                # comparison needs an explicit tol.
                v = value ** int(power)
                half = abs(v) * 1e-9
            else:
                v = value * scale10
                half = 0.5 * 10.0 ** -decimals * scale10
            v = -v if sign else v
            out.append((v, half))
            if extra:
                out.append((v * extra, half * extra))
    return out


def match(value, text):
    """The (value, half_unit) reading of `text` that `value` equals, or None."""
    for v, half in candidates(text):
        if math.isclose(value, v, rel_tol=1e-9, abs_tol=1e-300):
            return v, half
    return None


def well_anchored(text):
    """False for a citation too bare to identify one cell, like "1" or "2"."""
    t = norm(text)
    rest = re.sub(r"[\d.,\s\-+]", "", t)
    if rest:
        return True
    sig = re.sub(r"[^\d]", "", t).lstrip("0")
    return len(sig) >= 2


def _so(x):
    """Một con số để in: số nguyên có dấu phẩy ngăn nghìn, số thực giữ 6 chữ số có nghĩa."""
    if isinstance(x, (int, np.integer)) or (isinstance(x, float) and x.is_integer() and abs(x) < 1e15):
        return f"{int(x):,}"
    return f"{x:,.6g}"


def sach(gia_tri, nguon=None, trich=None, tol=None):
    """Một con số của sách, kèm đúng chữ của nó: trong bảng số liệu của bài học (nguon)
    hoặc nguyên văn trong chương (trich)."""
    chu = nguon if nguon is not None else trich
    if match(gia_tri, chu) is None:
        raise ValueError(f"{gia_tri!r} không phải con số ghi trong {chu!r}")
    return gia_tri


def theo_sach(ten, gia_tri, sach, nguon=None, trich=None, tol=None):
    """So một con số tính được với con số của sách. In ✓ khi khớp, ✗ kèm độ lệch khi không."""
    chu = nguon if nguon is not None else trich
    khop = match(sach, chu)
    if khop is None:
        raise ValueError(f"{sach!r} không phải con số ghi trong {chu!r}")
    sai_so = tol if tol is not None else khop[1]
    dung = abs(gia_tri - sach) <= sai_so
    print(f"{'✓' if dung else '✗'} {ten}: tính được {_so(gia_tri)} · sách: {chu}")
    if not dung:
        loi = f"lệch {_so(gia_tri - sach)}, quá sai số cho phép {_so(sai_so)}"
        if NB_STRICT:
            raise AssertionError(f"{ten}: {loi}")
        print(f"   {loi}. Nếu bạn vừa đổi tham số ở trên thì đây là điều được chờ đợi.")
    # None, not the bool: a cell ending in theo_sach(...) would otherwise echo True under the ✓.


def do_tren_may(ten, gia_tri, don_vi=""):
    """Một con số ĐO trên máy đang chạy: mỗi máy một khác, không phải con số của sách."""
    print(f"⏱ {ten}: {_so(gia_tri)} {don_vi} (đo trên máy này)".replace("  ", " "))


print(f"Python {sys.version.split()[0]} · PyTorch {torch.__version__} · NumPy {np.__version__}"
      f" · {torch.get_num_threads()} luồng CPU")

In [ ]:
import pandas as pd
import torch.nn as nn
import torchvision

MAU = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#8a5cc2"]  # thứ tự màu cố định
FP32, FP16 = 4, 2  # byte mỗi giá trị


def thoi_gian(f, lap=5):
    """Trung vị thời gian của f() (giây), sau một lần chạy khởi động."""
    f()
    do = []
    for _ in range(lap):
        t0 = time.perf_counter()
        f()
        do.append(time.perf_counter() - t0)
    return float(np.median(do))


print("sẵn sàng: torchvision", torchvision.__version__)

## 1. Tầng dày đặc: mỗi đầu vào nối với mỗi đầu ra

Perceptron nhiều tầng (multilayer perceptron, MLP) không giả định cấu trúc nào, nên mỗi đầu vào có một
trọng số riêng tới mỗi đầu ra. Tầng tham chiếu của chương nhận 784 giá trị, đúng số pixel của một ảnh
MNIST 28×28, và nối tới 100 nơ-ron ẩn. Mỗi nơ-ron đầu ra cộng dồn 784 tích, và để làm vậy nó phải đọc
784 đầu vào cùng 784 trọng số.

In [ ]:
vao = sach(784, nguon="784 · 100 · 78.400")
an = sach(100, nguon="784 · 100 · 78.400")
tang = nn.Linear(vao, an)

trong_so = vao * an
theo_sach("trọng số tầng 784 → 100", trong_so, sach=78_400, nguon="784 · 100 · 78.400")
assert tang.weight.numel() == trong_so  # PyTorch đếm đúng như vậy
print(f"cộng {tang.bias.numel()} hệ số điều chỉnh (bias) thì nn.Linear có "
      f"{sum(p.numel() for p in tang.parameters()):,} tham số; sách đếm riêng trọng số")

theo_sach("MAC cho mỗi nơ-ron đầu ra", trong_so // an, sach=784, nguon="784 MAC · 1.568 lần truy cập bộ nhớ")
theo_sach("lần truy cập bộ nhớ ít nhất cho mỗi nơ-ron (784 đầu vào + 784 trọng số)", vao + vao,
          sach=1568, nguon="784 MAC · 1.568 lần truy cập bộ nhớ")

### Dự đoán

Chương nói số tham số của tầng dày đặc tăng theo **bậc hai** của chiều rộng. Một tầng 1,024 → 1,024
cần khoảng một triệu tham số. Nếu gấp đôi cả hai chiều thành 2,048 → 2,048, số tham số gấp mấy lần?
Hãy đoán trước khi chạy ô dưới.

In [ ]:
def dem_day_dac(vao, ra, byte):
    """Số trọng số của tầng vao → ra, và số MB trọng số ở `byte` byte mỗi giá trị (1 MB = 10⁶ byte)."""
    return vao * ra, vao * ra * byte / 1e6


r1024 = sach(1024, nguon="1.024 → 1.024 · 1.048.576 · 2,1 MB FP16")
p1024, mb1024 = dem_day_dac(r1024, r1024, FP16)
theo_sach("tham số tầng 1,024 → 1,024", p1024, sach=1_048_576, nguon="1.048.576")
theo_sach("trọng số ở FP16 (MB)", mb1024, sach=2.1, nguon="2,1 MB FP16")

r2048 = sach(2048, nguon="2048 · 2048 · 4.194.304 · 16,8 MB FP32")
p2048, mb2048 = dem_day_dac(r2048, r2048, FP32)
theo_sach("tham số tầng 2,048 → 2,048", p2048, sach=4_194_304, nguon="4.194.304")
theo_sach("trọng số ở FP32 (MB)", mb2048, sach=16.8, nguon="16,8 MB FP32")

# Đếm không cần cấp phát: trên thiết bị "meta", PyTorch chỉ giữ hình dạng, không giữ dữ liệu.
assert nn.Linear(r2048, r2048, bias=False, device="meta").weight.numel() == p2048
print(f"chiều rộng gấp đôi, số tham số gấp {p2048 / p1024:.0f} lần")

Gấp đôi chiều rộng thì gấp bốn số tham số. Không có gì để chia sẻ: mỗi cặp đầu vào, đầu ra là một trọng
số riêng.

### Mật độ tính toán: bao nhiêu phép tính cho mỗi byte đọc về

Phương trình 2 của sách tính mật độ tính toán của tầng dày đặc ở batch 1 và FP32, theo một mô hình chỉ
đếm trọng số: mỗi trọng số cho hai phép tính (một nhân, một cộng) và cần đọc bốn byte.

$$I \approx \frac{2 \cdot M \cdot N \text{ FLOP}}{4 \cdot M \cdot N \text{ byte}}$$

$M$ và $N$ triệt tiêu, nên con số không phụ thuộc kích thước tầng. Ô dưới kiểm tra điều đó bằng phân
số chính xác.

In [ ]:
from fractions import Fraction

for M, N in ((784, 100), (1024, 1024), (2048, 2048), (12_288, 49_152)):
    assert Fraction(2 * M * N, FP32 * M * N) == Fraction(1, 2)
theo_sach("mật độ tính toán, batch 1, FP32, chỉ tính trọng số", 2 * 784 * 100 / (FP32 * 784 * 100),
          sach=0.5, nguon="0,5 FLOP/byte")
print("ở mọi kích thước tầng: I = 1/2 FLOP/byte")

Mô hình đó bỏ qua lưu lượng của đầu vào và đầu ra. Tính cả hai, và cho batch $B$ mẫu dùng chung một lần
đọc trọng số, thì mật độ của tầng $M \to N$ là

$$I(B) = \frac{2BMN}{4\,(MN + BM + BN)}.$$

Ở batch 1 nó thấp hơn 0.5 một chút; batch lớn dần thì nó tăng, rồi chững lại khi lưu lượng giá trị
kích hoạt lớn ngang trọng số. Chương nói nhiều bộ tăng tốc có điểm gãy (ridge point) ở mức hàng trăm
FLOP/byte; dải tô màu trên đồ thị là "hàng trăm" đó.

**Dự đoán:** tầng 2,048 → 2,048 cần batch cỡ nào để chạm tới dải đó?

In [ ]:
def mat_do(B, M, N, byte=FP32, ca_kich_hoat=True):
    """FLOP/byte của tầng M → N ở batch B: chỉ trọng số, hoặc cả đầu vào và đầu ra."""
    so_byte = byte * (M * N + (B * M + B * N if ca_kich_hoat else 0))
    return 2 * B * M * N / so_byte


cac_batch = [2**k for k in range(0, 13)]
fig, ax = plt.subplots(figsize=(7.5, 4.2))
ax.axhspan(100, 1000, color="0.9", lw=0)
ax.text(1.2, 300, "hàng trăm FLOP/byte:\nđiểm gãy của nhiều bộ tăng tốc", fontsize=8, color="0.35", va="center")
ax.loglog(cac_batch, [mat_do(b, r2048, r2048, ca_kich_hoat=False) for b in cac_batch], "o--",
          color=MAU[1], ms=4, base=2, label="chỉ tính trọng số (mô hình của phương trình 2)")
ax.loglog(cac_batch, [mat_do(b, r2048, r2048) for b in cac_batch], "o-", color=MAU[0], ms=5, base=2,
          label="tính cả đầu vào và đầu ra")
ax.annotate(f"batch 1: {mat_do(1, r2048, r2048):.3f}", (1, mat_do(1, r2048, r2048)), xytext=(18, -4),
            textcoords="offset points", fontsize=8, arrowprops=dict(arrowstyle="-", color="0.5", lw=0.8))
ax.set_xticks(cac_batch[::2], [str(b) for b in cac_batch[::2]])
moc_y = [0.5, 2, 8, 32, 128, 512, 2048]
ax.set_yticks(moc_y, [f"{v:g}" for v in moc_y])
ax.set_xlabel("kích thước batch")
ax.set_ylabel("mật độ tính toán (FLOP/byte, FP32)")
ax.set_title("Tầng dày đặc 2,048 → 2,048: batch dàn chi phí đọc trọng số")
ax.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()
can = next(b for b in cac_batch if mat_do(b, r2048, r2048) >= 100)
print(f"tính cả giá trị kích hoạt, tầng này chạm 100 FLOP/byte từ batch {can}; "
      f"trần của nó khi batch rất lớn là MN / (2(M + N)) = {r2048 * r2048 / (2 * (r2048 + r2048)):.0f}")

Đây là lý do một tầng dày đặc chạy batch 1 nghẽn ở bộ nhớ trên hầu hết bộ tăng tốc: nó cần cỡ nửa phép
tính cho mỗi byte, trong khi phần cứng cần hàng trăm. Batch là đòn bẩy duy nhất ở đây, và chính chương
lưu ý điểm chuyển còn tuỳ hình dạng ma trận, độ chính xác, cách hiện thực và phần cứng.

### Tầng dày đặc đặt lên ảnh

Với ảnh RGB 224×224, một tầng dày đặc cần một trọng số cho mỗi giá trị đầu vào, ở **mỗi** đơn vị đầu
ra. Bài học dùng con số này để mở phần "nghẽn ở đâu".

In [ ]:
canh = sach(224, nguon="27 · 224×224")
kenh_mau = 3  # R, G, B
moi_don_vi = canh * canh * kenh_mau
theo_sach("trọng số cho mỗi đơn vị đầu ra", moi_don_vi, sach=150_528, nguon="150.528 · ~150 triệu", tol=0)
TRAM_NAM_MUOI_TRIEU = sach(150e6, nguon="150.528 · ~150 triệu", tol=5e6)
print(f"~150 triệu trọng số ứng với {TRAM_NAM_MUOI_TRIEU / moi_don_vi:,.0f} đơn vị, tức một tầng chừng "
      f"một nghìn đơn vị; ở FP32 đó là {TRAM_NAM_MUOI_TRIEU * FP32 / 1e6:,.0f} MB phải đọc mỗi lần suy luận")

## 2. Tích chập: một bộ lọc nhỏ, dùng lại ở mọi vị trí

Mạng nơ-ron tích chập (convolutional neural network, CNN) giả định tính cục bộ không gian: một đầu ra
chỉ nhìn một vùng 3×3 quanh nó, và cùng một bộ lọc được dùng ở mọi vị trí. Đó là hai giả định, kết nối
cục bộ và chia sẻ tham số (parameter sharing), và chúng cắt số trọng số theo hai cách khác nhau.

In [ ]:
K = 3
bo_loc = K * K * kenh_mau
theo_sach("trọng số một bộ lọc 3×3 trên ảnh RGB", bo_loc, sach=27, nguon="27 · 224×224")
vi_tri = canh * canh
theo_sach("số vị trí mà bộ lọc được dùng lại", vi_tri, sach=50_176, nguon="50.176 lần")
theo_sach("ít hơn một bộ phát hiện đặc trưng dày đặc (lần)", moi_don_vi / bo_loc, sach=5575.1,
          trich="roughly 5,575.1× fewer parameters")
print("tỉ số đó là đúng 224² / 9: số vị trí chia cho số ô của bộ lọc")

Giờ là tầng tích chập mà chương dùng cho ImageNet: bộ lọc 3×3, 64 kênh vào, 64 kênh ra, trên ảnh
224×224. Có ba thứ để đếm: trọng số, giá trị kích hoạt (activation) mà tầng sinh ra, và số phép nhân
cộng dồn.

**Dự đoán:** cùng tầng đó trên ảnh 1024×1024 thì số trọng số đổi thế nào? Còn số giá trị kích hoạt?

In [ ]:
C = sach(64, nguon="224×224×64")
conv = nn.Conv2d(C, C, K, padding=1, bias=False)
w_conv = conv.weight.numel()
assert w_conv == K * K * C * C
theo_sach("trọng số tích chập 3×3, 64 → 64", w_conv, sach=37_000, trich="37,000\\) parameters regardless",
          tol=500)
day_dac_tuong_duong = vi_tri * C * C
theo_sach("tầng dày đặc tương đương trên 224×224×64", day_dac_tuong_duong, sach=205e6, nguon="~205 triệu",
          tol=1e6)
theo_sach("tỉ số dày đặc / tích chập", day_dac_tuong_duong / w_conv, sach=5575, nguon="~5.575×", tol=1)

# Trên thiết bị meta, một lượt truyền xuôi chỉ tính hình dạng: không tốn phép tính, không tốn bộ nhớ.
conv_meta = nn.Conv2d(C, C, K, padding=1, bias=False, device="meta")
y224 = conv_meta(torch.empty(1, C, 224, 224, device="meta"))
y1024 = conv_meta(torch.empty(1, C, 1024, 1024, device="meta"))
theo_sach("giá trị kích hoạt đầu ra ở 224×224", y224.numel(), sach=3.2e6, nguon="3,2 triệu giá trị kích hoạt")
print(f"ở 1024×1024: vẫn {conv_meta.weight.numel():,} trọng số, nhưng {y1024.numel():,} giá trị kích hoạt "
      f"({y1024.numel() / y224.numel():.1f} lần)")

mac_vi_tri_kenh = K * K * C  # mỗi vị trí, mỗi kênh đầu vào, cho cả 64 bộ lọc
theo_sach("MAC mỗi vị trí, mỗi kênh đầu vào", mac_vi_tri_kenh, sach=576, nguon="576 MAC · 50.176 vị trí")
mac_tang = mac_vi_tri_kenh * C * vi_tri
print(f"cả tầng, 64 kênh vào: {mac_tang:,} MAC = {mac_tang / 1e9:.2f} GMAC")
theo_sach("số lần mỗi trọng số được dùng (MAC / trọng số)", mac_tang / w_conv, sach=50_176, nguon="50.176 lần")

Số trọng số của tích chập không phụ thuộc kích thước ảnh: 36,864 ở 224×224 cũng như ở 1024×1024. Thứ
lớn lên theo ảnh là giá trị kích hoạt và số phép tính. Chương nói rõ điều này: tích chập tiết kiệm tham
số rất nhiều, nhưng có một loại bộ nhớ thứ hai nó không tiết kiệm.

Đồ thị dưới đặt ba đại lượng của tầng 64 → 64 cạnh nhau khi cạnh ảnh đi từ 28 tới 1024. Ở 224, ba điểm
được đánh dấu là ba con số của chương.

In [ ]:
cac_canh = [28, 56, 112, 224, 448, 1024]
fig, ax = plt.subplots(figsize=(7.5, 4.4))
ax.loglog(cac_canh, [c * c * C * C for c in cac_canh], "o-", color=MAU[1], label="trọng số tầng dày đặc tương đương")
ax.loglog(cac_canh, [c * c * C for c in cac_canh], "s-", color=MAU[2], label="giá trị kích hoạt đầu ra của tích chập")
ax.loglog(cac_canh, [w_conv] * len(cac_canh), "^-", color=MAU[0], label="trọng số tích chập 3×3")
for y, chu, lech in ((day_dac_tuong_duong, "~205 triệu", (10, -12)), (y224.numel(), "3.2 triệu", (8, -14)),
                     (w_conv, "~37 nghìn", (-20, 8))):
    ax.annotate(chu, (224, y), xytext=lech, textcoords="offset points", fontsize=8)
ax.set_xticks(cac_canh, [str(c) for c in cac_canh])
ax.minorticks_off()
ax.set_xlabel("cạnh ảnh (pixel), 64 kênh vào và 64 kênh ra")
ax.set_ylabel("số giá trị")
ax.set_title("Tích chập giữ số trọng số cố định; giá trị kích hoạt lớn theo ảnh")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=8)
plt.tight_layout()
plt.show()

## 3. Thêm một giả định: tách không gian khỏi kênh

Chuông cửa thông minh của chương phải phát hiện từ khoá (keyword spotting, KWS) trên một vi điều khiển
Cortex-M4 với 2 mW và 256 KB SRAM. Tích chập chuẩn ở đó quá đắt. Tích chập tách theo chiều sâu chia nó
làm hai: một bộ lọc 3×3 riêng cho từng kênh (phần không gian), rồi một tích chập điểm 1×1 trộn các kênh.

**Dự đoán:** với bộ lọc 3×3, 64 kênh vào và 128 kênh ra, cách tách giảm số trọng số bao nhiêu lần?

In [ ]:
C_vao = sach(64, nguon="64 → 128 kênh · 576")
C_ra = sach(128, nguon="64 → 128 kênh · 576")
chuan = nn.Conv2d(C_vao, C_ra, K, padding=1, bias=False)
theo_chieu_sau = nn.Conv2d(C_vao, C_vao, K, padding=1, groups=C_vao, bias=False)  # một bộ lọc mỗi kênh
diem = nn.Conv2d(C_vao, C_ra, 1, bias=False)

theo_sach("tích chập chuẩn", chuan.weight.numel(), sach=73_728, nguon="73.728 → 8.768")
theo_sach("tích chập theo chiều sâu", theo_chieu_sau.weight.numel(), sach=576, nguon="64 → 128 kênh · 576")
theo_sach("tích chập điểm 1×1", diem.weight.numel(), sach=8_192, nguon="576 · 8.192")
tach = theo_chieu_sau.weight.numel() + diem.weight.numel()
theo_sach("tách theo chiều sâu, tổng", tach, sach=8_768, nguon="73.728 → 8.768")
theo_sach("mức giảm", chuan.weight.numel() / tach, sach=8.4, nguon="~8,4×", tol=0.05)

x = torch.randn(1, C_vao, 16, 16)
assert chuan(x).shape == diem(theo_chieu_sau(x)).shape  # cùng hình dạng đầu ra
print("cùng đầu vào, cùng hình dạng đầu ra:", tuple(chuan(x).shape))

Cách tách không phải một phép xấp xỉ mơ hồ. Nó đúng bằng một tích chập chuẩn có trọng số mang một cấu
trúc riêng: trọng số nối kênh vào $c$ tới kênh ra $o$ là tích của một hệ số trộn kênh $P_{o,c}$ với bộ
lọc không gian $D_c$ của kênh đó. Ô dưới dựng tích chập chuẩn đó và kiểm rằng nó cho đúng kết quả của
hai tầng tách. Giả định "không gian và kênh xử lý rời được" chính là ràng buộc này.

In [ ]:
with torch.no_grad():
    D = theo_chieu_sau.weight[:, 0]                  # (64, 3, 3): một bộ lọc cho mỗi kênh
    P = diem.weight[:, :, 0, 0]                      # (128, 64): hệ số trộn kênh
    W_rang_buoc = P[:, :, None, None] * D[None]      # (128, 64, 3, 3)
    lech = torch.nn.functional.conv2d(x, W_rang_buoc, padding=1) - diem(theo_chieu_sau(x))
print(f"lệch lớn nhất giữa tích chập chuẩn có ràng buộc và hai tầng tách: {lech.abs().max():.1e}")
assert lech.abs().max() < 1e-4

# Mức giảm theo công thức của chương: phần còn lại là 1/C_ra + 1/K².
cac_C_ra = [16, 32, 64, 128, 256, 512, 1024]
giam = [1 / (1 / c + 1 / K**2) for c in cac_C_ra]
for c, g in zip(cac_C_ra, giam):
    assert math.isclose(g, (K * K * C_vao * c) / (K * K * C_vao + C_vao * c))
fig, ax = plt.subplots(figsize=(6.8, 3.6))
ax.semilogx(cac_C_ra, giam, "o-", color=MAU[0], base=2)
ax.axhline(K * K, color="0.5", lw=0.8, ls="--")
ax.text(16, K * K - 0.45, "giới hạn K² = 9 khi số kênh ra rất lớn", fontsize=8, color="0.35")
ax.annotate(f"128 kênh ra: {giam[3]:.1f}×", (128, giam[3]), xytext=(12, -24), textcoords="offset points",
            fontsize=8, arrowprops=dict(arrowstyle="-", color="0.5", lw=0.8))
ax.set_xticks(cac_C_ra, [str(c) for c in cac_C_ra])
ax.set_ylim(0, 10)
ax.set_xlabel("số kênh ra")
ax.set_ylabel("số lần ít trọng số hơn")
ax.set_title("Tách theo chiều sâu với bộ lọc 3×3")
plt.tight_layout()
plt.show()
theo_sach("mức giảm khi số kênh ra lớn, cận trên", K * K, sach=9, nguon="8–9×")

Vì sao chương nói tích chập chuẩn "quá đắt" cho chuông cửa? Một phép tính nhỏ của riêng sổ tay này, không
phải của sách: đặt trọng số của đúng một tầng đó cạnh 256 KB SRAM.

In [ ]:
SRAM_KB = sach(256, nguon="2 mW · 256 KB SRAM")
for ten, so in (("tích chập chuẩn", chuan.weight.numel()), ("tách theo chiều sâu", tach)):
    for kieu, byte in (("FP32", 4), ("INT8", 1)):
        print(f"{ten:>20}, {kieu}: {so * byte / 1024:7.1f} KiB = {100 * so * byte / (SRAM_KB * 1024):5.1f} % "
              f"của {SRAM_KB} KB SRAM")

Ở FP32, riêng trọng số của tầng chuẩn đã vượt cả 256 KB, dù đọc KB là 1,000 hay 1,024 byte. Bản tách
theo chiều sâu chiếm khoảng 13 %. Đó là chưa tính giá trị kích hoạt và các tầng khác.

## 4. Đếm hai ngọn hải đăng thật

Chương neo lập luận bằng năm mô hình "hải đăng". Hai trong số đó có sẵn trong torchvision. Gọi chúng với
`weights=None` thì không tải gì về, chỉ dựng kiến trúc với trọng số ngẫu nhiên, và thế là đủ để đếm. Ô
dưới gắn một hook vào mọi tầng tích chập và tuyến tính, chạy một ảnh 224×224, và ghi lại hình dạng của
từng tầng.

**Dự đoán:** trong ResNet-50, tích chập 3×3 hay tích chập 1×1 chiếm nhiều MAC hơn?

In [ ]:
def dem_tang(mo_hinh, canh=224, batch=1):
    """Một dòng cho mỗi tầng tích chập hoặc tuyến tính: loại, MAC, số trọng số, số giá trị vào và ra,
    cho cả batch (trọng số được đọc một lần cho cả batch)."""
    dong = []

    def hook(m, dau_vao, dau_ra):
        if isinstance(m, nn.Conv2d):
            k = m.kernel_size[0] * m.kernel_size[1]
            loai = "theo chiều sâu" if m.groups > 1 else f"{m.kernel_size[0]}×{m.kernel_size[1]}"
            mac = dau_ra.numel() * k * (m.in_channels // m.groups)
        else:
            loai, mac = "tuyến tính", dau_ra.numel() * m.in_features
        dong.append({"loại": loai, "MAC": mac, "trọng số": m.weight.numel(),
                     "giá trị vào": dau_vao[0].numel(), "giá trị ra": dau_ra.numel()})

    moc = [m.register_forward_hook(hook) for m in mo_hinh.modules() if isinstance(m, (nn.Conv2d, nn.Linear))]
    mo_hinh.eval()
    with torch.no_grad():
        mo_hinh(torch.zeros(batch, 3, canh, canh))
    for h in moc:
        h.remove()
    return pd.DataFrame(dong)


dat_hat_giong()
resnet = torchvision.models.resnet50(weights=None)
mobilenet = torchvision.models.mobilenet_v2(weights=None)
tang_rn, tang_mb = dem_tang(resnet), dem_tang(mobilenet)
tham_so_rn = sum(p.numel() for p in resnet.parameters())
tham_so_mb = sum(p.numel() for p in mobilenet.parameters())
gflop_rn, mflop_mb = 2 * tang_rn["MAC"].sum() / 1e9, 2 * tang_mb["MAC"].sum() / 1e6

theo_sach("tham số ResNet-50", tham_so_rn, sach=25.6e6, nguon="ResNet-50: 25,6M")
theo_sach("ResNet-50, GFLOP mỗi ảnh 224×224", gflop_rn, sach=8.2, nguon="25,6M · 8,2 GFLOP")
theo_sach("tham số MobileNetV2", tham_so_mb, sach=3.5e6, nguon="MobileNetV2: 3,5M")
theo_sach("MobileNetV2, MFLOP mỗi ảnh", mflop_mb, sach=600, nguon="3,5M · 14 MB · 600 MFLOP", tol=5)
print(f"đếm chính xác: ResNet-50 {tham_so_rn:,} tham số, {tang_rn['MAC'].sum():,} MAC; "
      f"MobileNetV2 {tham_so_mb:,} tham số, {tang_mb['MAC'].sum():,} MAC")

phan = tang_rn.groupby("loại")["MAC"].sum().sort_values(ascending=False)
print("ResNet-50, MAC theo loại tầng: " + ", ".join(f"{loai} {100 * v / phan.sum():.1f} %" for loai, v in phan.items()))

Hai con số khớp với sách. Còn phần của tích chập 3×3 thì sách ghi "khoảng 48 %", trong khi bản torchvision
cho thấp hơn. Lý do nằm ở một chi tiết hiện thực. Bản torchvision (thường gọi là ResNet-50 v1.5) đặt sải
bước (stride) 2 ở tích chập 3×3 của khối đầu mỗi tầng, nên tích chập 1×1 đứng trước nó chạy trên ánh xạ
đặc trưng (feature map) lớn gấp bốn. Bản gốc v1 đặt sải bước đó ở chính tích chập 1×1. Đổi lại cho đúng
bản gốc rồi đếm lại:

In [ ]:
resnet_v1 = torchvision.models.resnet50(weights=None)
for tang_lon in (resnet_v1.layer2, resnet_v1.layer3, resnet_v1.layer4):
    khoi = tang_lon[0]
    khoi.conv1.stride, khoi.conv2.stride = (2, 2), (1, 1)
tang_v1 = dem_tang(resnet_v1)
assert sum(p.numel() for p in resnet_v1.parameters()) == tham_so_rn  # cùng trọng số, khác chỗ đặt sải bước

for ten, bang in (("v1.5 (torchvision)", tang_rn), ("v1 (bản gốc)", tang_v1)):
    phan_3x3 = 100 * bang.loc[bang["loại"] == "3×3", "MAC"].sum() / bang["MAC"].sum()
    print(f"{ten:>19}: {2 * bang['MAC'].sum() / 1e9:.2f} GFLOP, tích chập 3×3 chiếm {phan_3x3:.1f} % MAC")
theo_sach("phần MAC của tích chập 3×3, bản v1", 100 * tang_v1.loc[tang_v1["loại"] == "3×3", "MAC"].sum()
          / tang_v1["MAC"].sum(), sach=48, trich="roughly 48% of MACs", tol=1)

Vậy hai con số của sách về cùng một "ResNet-50" khớp với hai biến thể khác nhau: 8.2 GFLOP khớp bản v1.5,
còn 48 % khớp bản v1, vốn chỉ cần khoảng 7.7 GFLOP. Cùng số tham số, cùng tên gọi, nhưng chỗ đặt một sải
bước đổi số phép tính khoảng 6 %. Một cái tên mô hình không cố định số FLOP của nó; hình dạng tensor mới
cố định.

### Chữ ký tải công việc và phổ mật độ tính toán

Bảng hải đăng của chương ghi tham số, FLOP và byte trọng số FP32 của từng mô hình, rồi chia FLOP cho byte
để ra một mật độ tính toán chỉ tính trọng số. Ô dưới làm lại phép chia đó, với số của sách và, khi có, với
số đếm được.

In [ ]:
RN_THAM_SO = sach(25.6e6, nguon="ResNet-50: 25,6M · 102,4 MB")
RN_FLOP = sach(8.2, nguon="25,6M · 8,2 GFLOP") * 1e9
MB_THAM_SO = sach(3.5e6, nguon="MobileNetV2: 3,5M")
MB_FLOP = sach(600, nguon="3,5M · 14 MB · 600 MFLOP") * 1e6
GPT_THAM_SO = sach(1.5e9, nguon="GPT-2 XL: 1,5B")
DLRM_THAM_SO = sach(25e9, nguon="DLRM: 25B · 100 GB")
KWS_THAM_SO = sach(200e3, trich="KWS (DS-CNN) | Audio | 200K")
KWS_FLOP = sach(20, trich="200K | 20 MFLOP") * 1e6

theo_sach("ResNet-50, MB trọng số FP32", RN_THAM_SO * FP32 / 1e6, sach=102.4, nguon="25,6M · 102,4 MB")
theo_sach("MobileNetV2, MB trọng số FP32", MB_THAM_SO * FP32 / 1e6, sach=14, nguon="3,5M · 14 MB")
theo_sach("GPT-2 XL, GB trọng số FP32", GPT_THAM_SO * FP32 / 1e9, sach=6, nguon="1,5B · 3 GFLOP/token · 6 GB")
theo_sach("GPT-2 XL, GFLOP mỗi token (2 FLOP mỗi tham số)", 2 * GPT_THAM_SO / 1e9, sach=3,
          nguon="1,5B · 3 GFLOP/token")
theo_sach("DLRM, GB trọng số FP32", DLRM_THAM_SO * FP32 / 1e9, sach=100, nguon="DLRM: 25B · 100 GB")
theo_sach("KWS, KB trọng số FP32", KWS_THAM_SO * FP32 / 1e3, sach=800, trich="20 MFLOP | 800 KB")

I_rn = RN_FLOP / (RN_THAM_SO * FP32)
I_gpt = 2 * GPT_THAM_SO / (GPT_THAM_SO * FP32)
theo_sach("ResNet-50, FLOP/byte", I_rn, sach=80.1, nguon="ResNet-50 ~80,1 FLOP/byte", tol=0.05)
theo_sach("GPT-2, FLOP/byte", I_gpt, sach=0.5, nguon="GPT-2 ~0,50 FLOP/byte", tol=0.005)
theo_sach("khoảng cách giữa hai đầu phổ", I_rn / I_gpt, sach=160.2, nguon="160,2×")
theo_sach("MobileNetV2 ít FLOP hơn ResNet-50 (lần)", RN_FLOP / MB_FLOP, sach=13.7, nguon="600 MFLOP · 13,7×")
theo_sach("MobileNetV2 nhỏ hơn ResNet-50 (lần)", RN_THAM_SO / MB_THAM_SO, sach=7.3, nguon="13,7× · 7,3×")

I_mb_tron = MB_FLOP / (MB_THAM_SO * FP32)
I_mb_dem = MB_FLOP / (tham_so_mb * FP32)
I_MB_SACH = sach(42.8, nguon="MobileNetV2 ~42,8 FLOP/byte", tol=0.05)
print(f"MobileNetV2: 600 MFLOP / 14 MB = {I_mb_tron:.2f} FLOP/byte; "
      f"600 MFLOP / ({tham_so_mb:,} × 4 byte) = {I_mb_dem:.2f}; sách: ~{I_MB_SACH}")

Phép chia cho MobileNetV2 không khép kín với số đã làm tròn: 600 chia 14 ra 42.86, tức 42.9 khi làm tròn,
còn sách ghi khoảng 42.8. Với số tham số chính xác mà torchvision đếm được, 3,504,872, phép chia ra đúng
42.80. Nhiều khả năng sách chia trên số chưa làm tròn; sổ tay in cả hai và không khẳng định con số nào.

Đặt năm hải đăng lên một bảng. Cột "điểm nghẽn" là của sách; mật độ của KWS là phép chia của riêng sổ tay
này, từ hai con số trong bảng của sách.

In [ ]:
bang = pd.DataFrame([
    ("ResNet-50", RN_THAM_SO, RN_FLOP, RN_THAM_SO * FP32, "tính toán"),
    ("MobileNetV2", MB_THAM_SO, MB_FLOP, MB_THAM_SO * FP32, "độ trễ"),
    ("KWS (DS-CNN)", KWS_THAM_SO, KWS_FLOP, KWS_THAM_SO * FP32, "điện năng"),
    ("GPT-2 XL, mỗi token", GPT_THAM_SO, 2 * GPT_THAM_SO, GPT_THAM_SO * FP32, "băng thông bộ nhớ"),
    ("DLRM", DLRM_THAM_SO, float("nan"), DLRM_THAM_SO * FP32, "dung lượng bộ nhớ"),
], columns=["mô hình", "tham số", "FLOP mỗi lượt", "byte trọng số FP32", "điểm nghẽn theo sách"])
bang["FLOP/byte"] = bang["FLOP mỗi lượt"] / bang["byte trọng số FP32"]

ve = bang.dropna().sort_values("FLOP/byte")
fig, ax = plt.subplots(figsize=(7.5, 3.4))
ax.barh(ve["mô hình"], ve["FLOP/byte"], color=[MAU[1], MAU[3], MAU[2], MAU[0]], height=0.55)
ax.set_xscale("log")
nhan = {"ResNet-50": "~80.1", "MobileNetV2": "~42.8", "GPT-2 XL, mỗi token": "~0.50", "KWS (DS-CNN)": "25 (tự tính)"}
for ten, gia_tri in zip(ve["mô hình"], ve["FLOP/byte"]):
    ax.text(gia_tri * 1.12, ten, nhan[ten], va="center", fontsize=8)
ax.set_xlim(0.2, 400)
ax.set_xlabel("FLOP chia cho byte trọng số FP32 (thang log)")
ax.set_title("Phổ mật độ tính toán của các hải đăng (mô hình chỉ tính trọng số)")
plt.tight_layout()
plt.show()
print(bang.to_string(index=False, na_rep="thấp", formatters={
    "tham số": "{:,.0f}".format, "FLOP mỗi lượt": "{:,.3g}".format, "byte trọng số FP32": "{:,.3g}".format,
    "FLOP/byte": lambda v: "thấp" if v != v else f"{v:.2f}"}))

DLRM không có mật độ trên phổ: sách ghi lượng tính toán của nó là "thấp", và 100 GB trọng số mới là chuyện
chính. Đó là kiểu điểm nghẽn thứ ba, dung lượng, mà một phép chia FLOP cho byte không nắm được.

### Mật độ của từng tầng, khi tính cả giá trị kích hoạt

Mô hình chỉ tính trọng số cho cả mô hình một con số. Chương nói thêm: MobileNet nằm giữa phổ ở mức toàn
mô hình, nhưng từng tầng tích chập theo chiều sâu tụt xuống thấp hơn nhiều khi tính cả lưu lượng giá trị
kích hoạt. Ô dưới tính mật độ của **từng tầng**, giả định mỗi tầng đọc đầu vào và trọng số của nó, ghi đầu
ra của nó, tất cả ở FP32 và không có gì nằm sẵn trong cache.

In [ ]:
for bang_tang in (tang_rn, tang_mb):
    bang_tang["FLOP/byte"] = 2 * bang_tang["MAC"] / (
        FP32 * (bang_tang["giá trị vào"] + bang_tang["trọng số"] + bang_tang["giá trị ra"]))

SAU_RESNET = sach(100, nguon="100–200+ FLOP/byte")
fig, ax = plt.subplots(figsize=(8, 4.4))
ax.axhspan(SAU_RESNET, 2 * SAU_RESNET, color="0.9", lw=0)
ax.text(1.2e5 * 2, 1.4 * SAU_RESNET, "100–200+ FLOP/byte: tầng sâu của ResNet theo sách", fontsize=8, color="0.35",
        va="center")
kieu = {"1×1": ("o", MAU[0]), "3×3": ("s", MAU[1]), "7×7": ("D", MAU[4]), "theo chiều sâu": ("^", MAU[2]),
        "tuyến tính": ("x", "0.3")}
for ten_mh, bang_tang, mo in (("ResNet-50", tang_rn, False), ("MobileNetV2", tang_mb, True)):
    for loai, nhom in bang_tang.groupby("loại"):
        dau, mau = kieu[loai]
        ax.scatter(nhom["MAC"], nhom["FLOP/byte"], marker=dau, s=26, color=mau,
                   facecolors="none" if mo and dau != "x" else mau,
                   label=f"{ten_mh}: {loai}")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("MAC của tầng (thang log)")
ax.set_ylabel("FLOP/byte, tính cả giá trị kích hoạt (thang log)")
ax.set_title("Mỗi điểm là một tầng (đặc: ResNet-50, rỗng: MobileNetV2), batch 1, FP32")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=7.5)
plt.tight_layout()
plt.show()
for ten_mh, bang_tang in (("ResNet-50", tang_rn), ("MobileNetV2", tang_mb)):
    print(f"{ten_mh}: trung vị {bang_tang['FLOP/byte'].median():.1f} FLOP/byte trên {len(bang_tang)} tầng; "
          + ", ".join(f"{loai} {n['FLOP/byte'].median():.1f}" for loai, n in bang_tang.groupby("loại")))

Trừ tầng tuyến tính cuối (ở batch 1 nó chính là tầng dày đặc 0.5 FLOP/byte của phần 1), các tầng theo
chiều sâu nằm thấp nhất trên đồ thị: mỗi kênh chỉ có 9 trọng số, nên mỗi giá trị đọc về chỉ được dùng cho
vài phép nhân. Đó chính là câu của chương: tích chập tách theo chiều sâu giảm tổng số phép tính, nhưng di
chuyển nhiều dữ liệu hơn so với lượng việc đó.

Dải xám là câu "các tầng bottleneck sâu hơn đạt 100–200+ FLOP/byte" của sách. Theo mô hình lưu lượng ở
đây, các tích chập 3×3 ở hai nhóm tầng đầu của ResNet-50 rơi vào dải đó, còn ở nhóm cuối, nơi ánh xạ đặc
trưng chỉ còn 7×7 mà mỗi tầng có 2.4 triệu trọng số, mật độ tụt xuống khoảng 24: ở batch 1, đọc trọng số
chiếm gần hết lưu lượng. Sách không nói con số của nó tính ở batch nào và theo mô hình lưu lượng nào, nên
hai cách đếm không nhất thiết trùng nhau. Mục "Thử thêm" cho bạn đổi batch để xem các tầng cuối dịch lên.

## 5. Ít FLOP hơn có nhanh hơn theo đúng tỉ lệ không?

Ngộ nhận của chương cho rằng MobileNetV2 ít FLOP hơn ResNet-50 13.7 lần, nên nó phải chạy nhanh hơn 13.7
lần. Sách kể một đội thay ResNet-50 bằng MobileNetV2 trên GPU A100 và thấy nó còn chậm hơn. Ở đây không có
GPU, nhưng có một câu hỏi đo được ngay trên CPU này.

**Dự đoán:** trên CPU này, ở batch 1, MobileNetV2 nhanh hơn ResNet-50 bao nhiêu lần?

In [ ]:
anh = torch.randn(1, 3, 224, 224)
with torch.no_grad():
    t_rn = thoi_gian(lambda: resnet(anh))
    t_mb = thoi_gian(lambda: mobilenet(anh))
do_tren_may("ResNet-50, một ảnh", 1000 * t_rn, "ms")
do_tren_may("MobileNetV2, một ảnh", 1000 * t_mb, "ms")
do_tren_may("tỉ số thời gian ResNet-50 / MobileNetV2", t_rn / t_mb, "lần")
print(f"tỉ số FLOP đếm được: {gflop_rn * 1e3 / mflop_mb:.1f} lần (sách: 13.7)")
do_tren_may("ResNet-50, FLOP/s đạt được", gflop_rn / t_rn, "GFLOP/s")
do_tren_may("MobileNetV2, FLOP/s đạt được", mflop_mb / 1e3 / t_mb, "GFLOP/s")
print(f"MAC trung bình mỗi tầng: ResNet-50 {tang_rn['MAC'].mean() / 1e6:.1f} triệu trên {len(tang_rn)} tầng, "
      f"MobileNetV2 {tang_mb['MAC'].mean() / 1e6:.1f} triệu trên {len(tang_mb)} tầng")

Nếu FLOP quyết định thời gian thì tỉ số thời gian phải gần bằng tỉ số FLOP. So hai con số in ra ở trên:
khoảng cách giữa chúng là phần mà FLOP không đo. Hai mô hình có số tầng gần bằng nhau, nhưng mỗi tầng của
MobileNetV2 nhỏ hơn hơn mười lần, nên phụ trội cố định (fixed overhead) của mỗi lần gọi kernel chiếm phần
lớn hơn, và các tầng theo chiều sâu, như phần trên vừa đếm, có mật độ tính toán thấp. FLOP/s đạt được của
hai mô hình vì thế thường khác nhau. Trên GPU lớn, khoảng cách đó còn rộng hơn, vì có nhiều đơn vị tính
toán hơn để mỗi kernel nhỏ phải lấp đầy. Đó là lý do chương kết luận phải đo trên đúng thiết bị đích, chứ
không suy ra từ số phép tính.

## Thử thêm

1. Ở ô đầu phần 4, đổi thành `tang_rn = dem_tang(resnet, batch=8)` rồi chạy lại ô mật độ từng tầng. Số MAC
   gấp 8 lần, lưu lượng trọng số thì không: các tích chập 3×3 của nhóm cuối dịch lên bao nhiêu? (Các dấu ✗
   về GFLOP khi đó là điều được chờ đợi.)
2. Ở phần 3, đổi `K = 5` (chạy lại từ ô phần 2). Mức giảm của tách theo chiều sâu tiến tới bao nhiêu khi
   số kênh ra lớn? (Các dấu ✗ khi đó là điều được chờ đợi: sách tính với bộ lọc 3×3.)
3. Thêm `torchvision.models.resnet18(weights=None)` hoặc `torchvision.models.mobilenet_v3_small(weights=None)`
   vào phần 4: chúng nằm đâu trên phổ mật độ tính toán, và đo được nhanh hơn ResNet-50 bao nhiêu lần?

## Tóm lại

* Tầng dày đặc có số tham số bằng tích của chiều vào và chiều ra, và ở batch 1 nó chỉ có khoảng 0.5 FLOP
  cho mỗi byte trọng số, bất kể kích thước; batch là đòn bẩy để nâng con số đó.
* Tích chập giữ số trọng số cố định theo kích thước ảnh và dùng lại mỗi trọng số ở mọi vị trí, nhưng giá
  trị kích hoạt và số phép tính vẫn lớn theo ảnh.
* Tách theo chiều sâu là một tích chập chuẩn có ràng buộc, giảm trọng số khoảng 8.4 lần ở cấu hình KWS,
  đổi lại các tầng có mật độ tính toán thấp.
* Đếm trên mô hình thật cho đúng con số của sách, và cho thấy một chi tiết hiện thực như chỗ đặt sải bước
  cũng đổi số FLOP.
* Tỉ số FLOP không phải tỉ số thời gian: hình dạng phép toán và phụ trội của từng kernel quyết định phần
  còn lại.

---
### Nguồn và giấy phép

Sổ tay này đi kèm bài học [Kiến trúc mạng](https://d3lu8vk4we0dc.cloudfront.net/lessons/vol1-ch06-nn-architectures/), dựng từ chương
[*Network Architectures*](https://mlsysbook.ai/vol1/nn_architectures/nn_architectures.html) của sách.

Nội dung gốc thuộc Machine Learning Systems của Vijay Janapa Reddi và cộng sự, MIT Press. Bản quyền © 2024-2026 Harvard University. Giấy phép CC BY-NC-SA 4.0. Bản tiếng Việt này là tác phẩm phái sinh dùng cùng giấy phép: ghi công, chia sẻ tương tự, phi thương mại.